# Pandas로 표 다루기

주문 표에서 결제 완료 행만 골라 재사용 가능한 반환값을 만듭니다.

## 실행 환경 준비

이 노트북은 셀 안에서 교육용 주문 데이터를 준비합니다. `import pandas as pd`는 pandas를 `pd`라는 이름으로 불러옵니다. 이후 `pd.DataFrame(...)`과 `pd.read_csv(...)`의 점은 pandas 안의 생성자와 함수를 선택하는 표기입니다.

In [1]:
# 표 계산에 사용할 pandas와 임시 파일 경로를 준비합니다.
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

print("pandas", pd.__version__)

pandas 3.0.5


## 기본 실행

`pd.DataFrame(사전 목록)`은 사전의 키를 열 이름으로 사용한 표를 반환합니다. `to_csv`는 표를 CSV 파일에 쓰고 `pd.read_csv(경로)`는 첫 줄을 열 이름으로 읽어 새 DataFrame을 반환합니다. 열 이름 하나를 대괄호에 쓰면 인덱스와 값을 함께 가진 1차원 `Series`를 얻습니다. 이 셀에서는 사전 세 개를 3행 3열 표로 바꾸고 열별 자료형을 확인합니다.

`to_string(index=False)`는 행 인덱스를 생략한 표 문자열을 만듭니다. `orders.dtypes`는 열마다 자료형을 담은 Series이고, 여기에 `astype(str)`를 적용하면 자료형 이름을 문자열로 바꾼 새 Series를 얻습니다. 마지막 `to_dict()`는 열 이름과 자료형 이름을 짝지은 사전으로 바꿉니다.

In [2]:
# 노트북 안에서 CSV 파일을 만들어 파일 입력도 독립적으로 재현합니다.
temp_dir = TemporaryDirectory()
orders_path = Path(temp_dir.name) / "orders.csv"
pd.DataFrame([
    {"order_id":"O101", "status":"paid", "amount":32000},
    {"order_id":"O102", "status":"pending", "amount":18000},
    {"order_id":"O103", "status":"paid", "amount":27000},
]).to_csv(orders_path, index=False)
orders = pd.read_csv(orders_path, dtype={"order_id":"string"})
print(orders.to_string(index=False))
print(orders.dtypes.astype(str).to_dict())

order_id  status  amount
    O101    paid   32000
    O102 pending   18000
    O103    paid   27000
{'order_id': 'string', 'status': 'str', 'amount': 'int64'}


## 중간 결과 관찰

`frame["status"]`는 상태 열의 `Series`입니다. 이 Series를 입력 상태와 비교하면 행마다 참·거짓인 불리언 마스크가 만들어집니다. `loc`는 그 마스크가 참인 행을 선택해 새 DataFrame을 반환하고, `to_dict(orient="records")`는 각 행을 사전으로 바꿔 사전 목록을 반환합니다.

Ch 2의 07절에서 배운 타입 힌트에 여러 허용 형식을 표시합니다. `str | Path`는 문자열 경로나 Path 객체를 받을 수 있다는 뜻입니다. 홀로 놓인 `*` 뒤의 `status`는 `status="paid"`처럼 이름을 붙여 전달합니다. `str | None = None`은 문자열 또는 값 없음을 허용하며, 생략하면 None으로 두어 모든 상태를 읽습니다. `-> list[dict]`는 사전을 항목으로 담은 목록을 반환한다는 표시입니다.

`tolist()`는 Series의 값들을 Python 목록으로 반환합니다. `[row["order_id"] for row in paid]`는 `paid`를 한 항목씩 `row`로 읽어 주문 번호만 새 목록에 모으는 표기입니다. 대괄호 안의 `for`를 이렇게 읽으면 검사 대상이 주문 번호 목록임을 알 수 있습니다.

In [3]:
# Series 비교가 행마다 만드는 불리언 마스크를 먼저 확인합니다.
status_series = orders["status"]
paid_mask = status_series == "paid"
print("상태 Series:", status_series.tolist())
print("paid 마스크:", paid_mask.tolist())

# 파일 경로와 선택 상태를 받는 재사용 함수로 반환 형식을 고정합니다.
def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict]:
    frame = pd.read_csv(csv_path, dtype={"order_id":"string"})
    if status is not None:
        frame = frame.loc[frame["status"] == status]
    return frame.to_dict(orient="records")
paid = fetch_orders(orders_path, status="paid")
print(paid)
assert [row["order_id"] for row in paid] == ["O101", "O103"]

상태 Series: ['paid', 'pending', 'paid']
paid 마스크: [True, False, True]
[{'order_id': 'O101', 'status': 'paid', 'amount': 32000}, {'order_id': 'O103', 'status': 'paid', 'amount': 27000}]


## 값 하나를 바꿔 비교

입력 데이터와 처리 순서는 유지하고 상태 인자만 `paid`에서 `pending`으로 바꿉니다. 반환 목록에서 O102 한 건만 남는지 확인합니다.

In [4]:
# 필터 값만 바꾸어 같은 함수가 다른 상태를 선택하는지 확인합니다.
pending = fetch_orders(orders_path, status="pending")
print(pending)
assert [row["order_id"] for row in pending] == ["O102"]

[{'order_id': 'O102', 'status': 'pending', 'amount': 18000}]


## 실패 재현

금액이 문자열이면 합계가 숫자 덧셈이 아닌 문자열 연결이 됩니다.

`orders.astype({"amount": "string"})`는 amount 열을 문자열로 바꾼 새 DataFrame을 반환합니다. 반환값을 `broken`에 저장하므로 원래 `orders`의 숫자 열은 그대로 두고 두 경우를 비교할 수 있습니다.

In [5]:
# 금액을 문자열로 읽으면 sum이 문자열 연결을 수행합니다.
broken = orders.astype({"amount":"string"})
wrong_total = broken["amount"].sum()
print("문자열 합계:", wrong_total)
assert wrong_total == "320001800027000"

문자열 합계: 320001800027000


## 수정

숫자로 변환한 새 열을 사용해 합계를 계산합니다.

`assign(amount=...)`은 오른쪽 계산 결과로 amount 열을 교체한 새 DataFrame을 반환합니다. 먼저 `pd.to_numeric`이 문자열 금액을 숫자 Series로 변환하고, `assign`이 그 Series를 금액 열로 사용합니다. 원래 `broken`을 직접 고친 것이 아니라 새 표를 `fixed`에 저장합니다.

In [6]:
# 숫자 변환 실패를 드러내고 정수 합계를 다시 계산합니다.
fixed = broken.assign(amount=pd.to_numeric(broken["amount"], errors="raise"))
correct_total = int(fixed["amount"].sum())
print("숫자 합계:", correct_total)
assert correct_total == 77000
temp_dir.cleanup()

숫자 합계: 77000
